# Instituto Tecnológico de Costa Rica
## Escuela de Computación • Curso de Inteligencia Artificial
### Proyecto Final: Reconocimiento Automático del Habla (ASR) con Pérdida CTC
---
**Estudiantes:** Joshua Picado & Carlos Arce  
**Profesor:** Ing. Efrén Jiménez Delgado  
**Semestre:** II Semestre, 2026  

**Objetivo del Cuaderno:**  
Entrenar y evaluar un modelo acústico neuronal (**CNN 2D + Bi-GRU estilo DeepSpeech2**) optimizado mediante la función de pérdida **Connectionist Temporal Classification (CTC Loss)** sobre un corpus de 200 audios normalizados (WAV mono a 16 kHz) grabados por dos locutores independientes. Se evalúa el rendimiento con las métricas **WER (Word Error Rate)** y **CER (Character Error Rate)**.

## PASO 1: Conexión a Google Drive e Instalación de Dependencias

In [9]:
# 1. Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Instalar jiwer para métricas WER y CER
!pip install -q jiwer

import os
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from jiwer import wer, cer

print(f"\n[INFO] TensorFlow Version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"[INFO] Acelerador GPU activo: {gpus[0].name}")
else:
    print("[ADVERTENCIA] Active GPU T4 en Entorno de ejecución -> Cambiar tipo de entorno.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

[INFO] TensorFlow Version: 2.21.0
[INFO] Acelerador GPU activo: /physical_device:GPU:0


## PASO 2: Carga y Validación del Dataset de Audio (200 Muestras WAV a 16 kHz)

In [10]:
base_drive = '/content/drive/MyDrive' if os.path.exists('/content/drive/MyDrive') else '/content/drive/My Drive'
IA_DIR = os.path.join(base_drive, 'IA')
DATASETS_DIR = os.path.join(IA_DIR, 'Datasets')

# Localizar carpeta o archivo zip de audio
candidate_audio = [
    os.path.join(DATASETS_DIR, 'audio_dataset'),
    os.path.join(DATASETS_DIR, 'audio'),
    DATASETS_DIR
]

AUDIO_BASE = None
for p in candidate_audio:
    if os.path.exists(os.path.join(p, 'metadata.csv')):
        AUDIO_BASE = p
        break

if AUDIO_BASE is None:
    zip_candidates = ['dataset_audio_200.zip', 'dataset_audio.zip']
    for zname in zip_candidates:
        zpath = os.path.join(DATASETS_DIR, zname)
        if os.path.exists(zpath):
            print(f"[INFO] Descomprimiendo {zpath}...")
            !unzip -q "{zpath}" -d "{DATASETS_DIR}"
            AUDIO_BASE = os.path.join(DATASETS_DIR, 'audio_dataset')
            break

if AUDIO_BASE is None or not os.path.exists(os.path.join(AUDIO_BASE, 'metadata.csv')):
    raise FileNotFoundError(f"No se encontró 'metadata.csv' en {DATASETS_DIR}")

CSV_PATH = os.path.join(AUDIO_BASE, 'metadata.csv')
WAVS_DIR = os.path.join(AUDIO_BASE, 'wavs') if os.path.exists(os.path.join(AUDIO_BASE, 'wavs')) else AUDIO_BASE

df = pd.read_csv(CSV_PATH)
print(f"\n[ÉXITO] Corpus de audio cargado: {len(df)} grabaciones")
print(df.head())

# Normalización de transcripción a minúsculas y caracteres limpios
df['transcription'] = df['transcription'].astype(str).str.lower().str.strip()
df['file_path'] = df['audio_file'].apply(lambda x: os.path.join(WAVS_DIR, x) if not os.path.isabs(x) else x)

# Partición 80% Train / 20% Validación con semilla fija
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
split_idx = int(len(df) * 0.8)
df_train = df[:split_idx]
df_val = df[split_idx:]

print(f"Partición de Audio: {len(df_train)} muestras Train (80%) | {len(df_val)} muestras Val (20%)")


[ÉXITO] Corpus de audio cargado: 200 grabaciones
       audio_file    transcription
0  audio_0001.wav  activar sistema
1  audio_0002.wav    detener video
2  audio_0003.wav  analizar imagen
3  audio_0004.wav  palma detectada
4  audio_0005.wav   puño detectado
Partición de Audio: 160 muestras Train (80%) | 40 muestras Val (20%)


## PASO 3: Vocabulario Fonético y Capas de Mapeo de Caracteres

In [11]:
# Construir vocabulario a partir de todos los caracteres presentes
all_chars = set("".join(df['transcription'].tolist()))
characters = sorted(list(all_chars))
print(f"Vocabulario fonético ({len(characters)} caracteres): {characters}")

# Mapeo de caracteres con StringLookup
char_to_num = keras.layers.StringLookup(vocabulary=characters, oov_token="")
num_to_char = keras.layers.StringLookup(
    vocabulary=char_to_num.get_vocabulary(), oov_token="", invert=True
)

# Exportar vocabulario en JSON para FastAPI
vocab_out = os.path.join(IA_DIR, 'ctc_vocab.json')
with open(vocab_out, 'w', encoding='utf-8') as f:
    json.dump(characters, f, ensure_ascii=False)
print(f"Vocabulario exportado a: {vocab_out}")

Vocabulario fonético (25 caracteres): [' ', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'y', 'z', 'ñ']
Vocabulario exportado a: /content/drive/MyDrive/IA/ctc_vocab.json


## PASO 4: Preprocesamiento de Audio: Espectrograma STFT (Igual a la clase)

In [12]:
frame_length = 256
frame_step = 160
fft_length = 384

def encode_single_sample(wav_file, label):
    file = tf.io.read_file(wav_file)
    audio, _ = tf.audio.decode_wav(file, desired_channels=1)
    audio = tf.squeeze(audio, axis=-1)
    audio = tf.cast(audio, tf.float32)

    # STFT y magnitud espectral
    spectrogram = tf.signal.stft(
        audio, frame_length=frame_length, frame_step=frame_step, fft_length=fft_length
    )
    spectrogram = tf.abs(spectrogram)
    spectrogram = tf.math.pow(spectrogram, 0.5)

    # Normalización
    means = tf.math.reduce_mean(spectrogram, 1, keepdims=True)
    stddevs = tf.math.reduce_std(spectrogram, 1, keepdims=True)
    spectrogram = (spectrogram - means) / (stddevs + 1e-10)

    # Mapeo de etiqueta textual
    label = tf.strings.lower(label)
    label = tf.strings.unicode_split(label, input_encoding="UTF-8")
    label = char_to_num(label)
    return spectrogram, label

BATCH_SIZE = 8  # Lote balanceado para gradientes estables en CTC
train_ds = tf.data.Dataset.from_tensor_slices((list(df_train["file_path"]), list(df_train["transcription"])))
train_ds = train_ds.map(encode_single_sample, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.padded_batch(BATCH_SIZE).prefetch(buffer_size=tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((list(df_val["file_path"]), list(df_val["transcription"])))
val_ds = val_ds.map(encode_single_sample, num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds.padded_batch(BATCH_SIZE).prefetch(buffer_size=tf.data.AUTOTUNE)

print(f"Pipelines listos con BATCH_SIZE = {BATCH_SIZE}")

Pipelines listos con BATCH_SIZE = 8


## PASO 5: Función de Pérdida CTC y Modelo Acústico Neuronal
Se ajusta la capacidad del modelo (2 capas Bi-GRU de 128 unidades) y una tasa de aprendizaje óptima ($8 \times 10^{-4}$) para que la red converja rápidamente sobre el corpus acotado sin quedarse atrapada en el colapso de predecir únicamente *blanks*.

In [13]:
def CTCLoss(y_true, y_pred):
    batch_len = tf.cast(tf.shape(y_true)[0], dtype="int64")
    input_length = tf.cast(tf.shape(y_pred)[1], dtype="int64")
    label_length = tf.cast(tf.shape(y_true)[1], dtype="int64")
    input_length = input_length * tf.ones(shape=(batch_len, 1), dtype="int64")
    label_length = label_length * tf.ones(shape=(batch_len, 1), dtype="int64")
    loss = keras.backend.ctc_batch_cost(y_true, y_pred, input_length, label_length)
    return loss

def build_ctc_model(input_dim, output_dim, rnn_layers=2, rnn_units=128):
    input_spectrogram = layers.Input((None, input_dim), name="input_spectrogram")
    x = layers.Reshape((-1, input_dim, 1), name="expand_dim")(input_spectrogram)

    # Bloque Convolucional 1 (Submuestreo temporal x2)
    x = layers.Conv2D(32, [11, 41], strides=[2, 2], padding="same", use_bias=False, name="conv_1")(x)
    x = layers.BatchNormalization(name="conv_1_bn")(x)
    x = layers.ReLU(name="conv_1_relu")(x)

    # Bloque Convolucional 2 (Preserva dimensión temporal)
    x = layers.Conv2D(32, [11, 21], strides=[1, 2], padding="same", use_bias=False, name="conv_2")(x)
    x = layers.BatchNormalization(name="conv_2_bn")(x)
    x = layers.ReLU(name="conv_2_relu")(x)

    # Aplanar dimensiones para alimentar las capas recurrentes
    x = layers.Reshape((-1, x.shape[-2] * x.shape[-1]))(x)

    # Capas Recurrentes Bidireccionales (GRU)
    for i in range(1, rnn_layers + 1):
        recurrent = layers.GRU(
            units=rnn_units, activation="tanh", recurrent_activation="sigmoid",
            use_bias=True, return_sequences=True, reset_after=True, name=f"gru_{i}"
        )
        x = layers.Bidirectional(recurrent, name=f"bidirectional_{i}", merge_mode="concat")(x)
        if i < rnn_layers:
            x = layers.Dropout(rate=0.25)(x)

    # Capa Densa Final de Proyección
    x = layers.Dense(units=rnn_units, name="dense_proj")(x)
    x = layers.ReLU(name="dense_relu")(x)
    x = layers.Dropout(rate=0.25)(x)
    output = layers.Dense(units=output_dim + 1, activation="softmax", name="ctc_output")(x)

    model = keras.Model(input_spectrogram, output, name="DeepSpeech2_Compact")
    # Tasa de aprendizaje óptima para escapar del mínimo local de blanks
    opt = keras.optimizers.Adam(learning_rate=8e-4)
    model.compile(optimizer=opt, loss=CTCLoss)
    return model

model_ctc = build_ctc_model(
    input_dim=fft_length // 2 + 1,
    output_dim=char_to_num.vocabulary_size(),
    rnn_layers=2,
    rnn_units=128
)
model_ctc.summary()

Model: "DeepSpeech2_Compact"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_spectrogram (InputLayer)  │ (None, None, 193)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ expand_dim (Reshape)            │ (None, None, 193, 1)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_1 (Conv2D)                 │ (None, None, 97, 32)   │        14,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_1_bn (BatchNormalization)  │ (None, None, 97, 32)   │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_1_relu (ReLU)              │ (None, None, 97, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_2 (Conv2D)                 │ (None, None, 49, 32)   │       236,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_2_bn (BatchNormalization)  │ (None, None, 49, 32)   │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_2_relu (ReLU)              │ (None, None, 49, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape_1 (Reshape)             │ (None, None, 1568)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, None, 256)      │     1,304,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, None, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, None, 256)      │       296,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_proj (Dense)              │ (None, None, 128)      │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_relu (ReLU)               │ (None, None, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, None, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ctc_output (Dense)              │ (None, None, 27)       │         3,483 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,888,123 (7.20 MB)

 Trainable params: 1,887,995 (7.20 MB)

 Non-trainable params: 128 (512.00 B)

## PASO 6: Entrenamiento del Modelo Acústico CTC (35 Épocas)
Permite que la red atraviese la meseta inicial y aprenda las transiciones fonéticas reales.

In [14]:
EPOCHS = 35
callbacks = [
    keras.callbacks.EarlyStopping(monitor="loss", patience=12, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor="loss", factor=0.7, patience=5, verbose=1)
]

print("\nIniciando entrenamiento del modelo acústico CTC...")
history_ctc = model_ctc.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)


Iniciando entrenamiento del modelo acústico CTC...
Epoch 1/35
20/20 ━━━━━━━━━━━━━━━━━━━━ 27s 924ms/step - loss: 62.3963 - val_loss: 49.3106 - learning_rate: 8.0000e-04
Epoch 2/35
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 104ms/step - loss: 44.8908 - val_loss: 46.9617 - learning_rate: 8.0000e-04
Epoch 3/35
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 80ms/step - loss: 43.5875 - val_loss: 46.4120 - learning_rate: 8.0000e-04
Epoch 4/35
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 79ms/step - loss: 42.5970 - val_loss: 46.1037 - learning_rate: 8.0000e-04
Epoch 5/35
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 86ms/step - loss: 41.8174 - val_loss: 45.9566 - learning_rate: 8.0000e-04
Epoch 6/35
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 85ms/step - loss: 41.1133 - val_loss: 44.6205 - learning_rate: 8.0000e-04
Epoch 7/35
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 81ms/step - loss: 40.2104 - val_loss: 42.9474 - learning_rate: 8.0000e-04
Epoch 8/35
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 98ms/step - loss: 39.4470 - val_loss: 41.7919 - learning_rate: 8.0000e-04
Epoch 9/35
20/20 ━━━━━━━━

## PASO 7: Evaluación con WER, CER y Decodificador Voraz (Greedy Decode)
Se decodifican los fotogramas temporales colapsando repeticiones y removiendo los tokens *blank* (-1) para obtener las palabras transcritas.

In [15]:
def decode_batch_predictions(pred):
    input_len = np.ones(pred.shape[0]) * pred.shape[1]
    results = keras.backend.ctc_decode(pred, input_length=input_len, greedy=True)[0][0]
    output_text = []
    for result in results:
        # Filtrar explícitamente los índices negativos (-1) generados por los blanks de CTC
        valid_mask = result >= 0
        valid_indices = tf.boolean_mask(result, valid_mask)
        text = tf.strings.reduce_join(num_to_char(valid_indices)).numpy().decode("utf-8")
        output_text.append(text)
    return output_text

predictions = []
targets = []
latencies = []

for batch in val_ds:
    X, y = batch
    t0 = time.perf_counter()
    batch_pred = model_ctc.predict(X, verbose=0)
    t1 = time.perf_counter()
    latencies.append((t1 - t0) / len(X))

    batch_pred = decode_batch_predictions(batch_pred)
    predictions.extend(batch_pred)
    for label in y:
        valid_mask = label >= 0
        valid_label = tf.boolean_mask(label, valid_mask)
        target_text = tf.strings.reduce_join(num_to_char(valid_label)).numpy().decode("utf-8")
        targets.append(target_text)

wer_score = wer(targets, predictions)
cer_score = cer(targets, predictions)
avg_lat = np.mean(latencies)

print("\n" + "="*55)
print("  RESULTADOS FORMALES DEL MODELO ACÚSTICO CTC")
print("="*55)
print(f"WER (Word Error Rate):       {wer_score * 100:.2f}%")
print(f"CER (Character Error Rate):  {cer_score * 100:.2f}%")
print(f"Tiempo medio de inferencia:  {avg_lat:.3f} s por audio (< 5 s requerido)")

print("\n--- Ejemplos de Transcripción (Real vs Predicción CTC) ---")
for i in range(min(12, len(predictions))):
    print(f"Real:       '{targets[i]}'")
    print(f"Predicción: '{predictions[i]}'\n")


  RESULTADOS FORMALES DEL MODELO ACÚSTICO CTC
WER (Word Error Rate):       84.15%
CER (Character Error Rate):  44.15%
Tiempo medio de inferencia:  0.026 s por audio (< 5 s requerido)

--- Ejemplos de Transcripción (Real vs Predicción CTC) ---
Real:       'reproducir video'
Predicción: 'recor eo'

Real:       'cambiar modelo'
Predicción: 'cabiar o'

Real:       'bajar volumen'
Predicción: 'bajar '

Real:       'guardar captura'
Predicción: 'puñar a'

Real:       'palma detectada'
Predicción: 'palma cta'

Real:       'puño detectado'
Predicción: 'puño ctad'

Real:       'detener video'
Predicción: 'deter eo'

Real:       'activar sistema'
Predicción: 'actiar sitema'

Real:       'pausar audio'
Predicción: 'pusar a'

Real:       'modo nocturno'
Predicción: 'modo '

Real:       'modo nocturno'
Predicción: 'moudo '

Real:       'modo nocturno'
Predicción: 'modo '



## PASO 8: Exportación del Modelo Acústico Entrenado a Google Drive

In [16]:
model_out = os.path.join(IA_DIR, 'audio_ctc_model.keras')
model_ctc.save(model_out)
print(f"\n[ÉXITO] Modelo acústico CTC guardado en su Drive: {model_out}")


[ÉXITO] Modelo acústico CTC guardado en su Drive: /content/drive/MyDrive/IA/audio_ctc_model.keras
Copie este modelo junto a 'ctc_vocab.json' en 'backend/models_dl/' de su proyecto.
